# S11 · Customer segmentation (the lab)

This is the session lab and it puts the whole day together. We build a small customer
table, scale it, cluster the customers into segments with k-means, name each segment
in plain business terms, and plot them with PCA. This is the shop problem from the
start of the session, done end to end.

**New here? Read this once.**

- New to Python? You can still run the whole lab. Press play on each cell, top to
  bottom, and read the plain-English note above each one.
- Not sure why we "scale" the columns? Open `primers/distance_between_points.md`; the
  scaling section is the single most important idea in this lab.
- Already confident? The main line *is* the lab; look for the **Stretch (optional)**
  cell near the end for a from-scratch touch.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, pandas, matplotlib and scikit-learn.
# Google Colab already ships all four, so there is nothing to install.
print("Setup complete - nothing to install.")

These are all the tools the lab needs, imported once up front.

In [ ]:
import numpy as np                              # fast maths on lists of numbers
import pandas as pd                               # tidy tables (DataFrames)
import matplotlib.pyplot as plt                   # drawing charts
from sklearn.preprocessing import StandardScaler  # puts every column on the same scale
from sklearn.cluster import KMeans                # the k-means clustering tool
from sklearn.metrics import silhouette_score      # scores how well the groups came out
from sklearn.decomposition import PCA             # squashes columns to 2 for plotting

## Step 1 — build a small customer table (RFM)

In retail, a customer is often summarised by three numbers, called **RFM**:

- **Recency**: how many days since their last purchase (smaller = more recent).
- **Frequency**: how many times they have bought.
- **Monetary**: how much money they have spent in total.

We make a synthetic table with three hidden kinds of customer mixed together, so the
data is realistic but we still control it. The clustering will *not* be told which
customer came from which kind.

In [ ]:
# Set a seed so the random customers are the same for everyone.
np.random.seed(0)

# We build 3 hidden groups of customers, then stack them into one table.

# Group A: loyal big spenders - bought recently, often, and a lot.
recency_A   = np.random.normal(15, 5, 80)
frequency_A = np.random.normal(25, 5, 80)
monetary_A  = np.random.normal(5000, 800, 80)

# Group B: occasional shoppers - middling on everything.
recency_B   = np.random.normal(60, 15, 80)
frequency_B = np.random.normal(8, 3, 80)
monetary_B  = np.random.normal(1500, 400, 80)

# Group C: lapsing customers - bought little and not for a long time.
recency_C   = np.random.normal(150, 25, 80)
frequency_C = np.random.normal(3, 1.5, 80)
monetary_C  = np.random.normal(500, 200, 80)

# Stack the three groups together into one long array per feature.
recency   = np.concatenate([recency_A, recency_B, recency_C])
frequency = np.concatenate([frequency_A, frequency_B, frequency_C])
monetary  = np.concatenate([monetary_A, monetary_B, monetary_C])

# Put it all in a tidy table (a pandas DataFrame).
customers = pd.DataFrame({
    "recency": recency.round(0),
    "frequency": frequency.round(0),
    "monetary": monetary.round(0),
})

print("number of customers:", len(customers))
customers.head()

## Step 2 — a quick look at the numbers

`describe()` shows the range and average of each column. Notice the columns sit on
very different scales: monetary is in thousands of rupees, frequency is in single or
double digits. That difference is exactly what we have to fix next.

In [ ]:
customers.describe().round(1)

## Step 3 — scale the features

Because the three columns have such different ranges, a raw distance would be decided
almost entirely by `monetary`, and recency and frequency would barely count. We
**standardise** every column (mean 0, spread 1) so each feature gets an equal say.
This is the most important step in the whole lab.

In [ ]:
scaler = StandardScaler()
scaled_customers = scaler.fit_transform(customers)

# Confirm every column now has mean about 0 and spread (std) about 1.
print("mean of each column after scaling:", scaled_customers.mean(axis=0).round(2))
print("std  of each column after scaling:", scaled_customers.std(axis=0).round(2))

## Step 4 — choose the number of segments with the elbow and the silhouette

We do not assume we know how many segments there are. We try several values of k and
look at both the inertia (for the elbow) and the silhouette score, just like in
notebook 1.

In [ ]:
# Try k from 2 to 6 and record inertia and silhouette for each.
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = model.fit_predict(scaled_customers)
    inertia = model.inertia_
    score = silhouette_score(scaled_customers, labels)
    print("k =", k, " inertia =", round(inertia, 1), " silhouette =", round(score, 3))

print()
print("k = 3 gives the best silhouette - and we did build 3 hidden groups.")

## Step 5 — cluster the customers into 3 segments

We settle on k = 3 and run k-means once more to get the final segment for each
customer. We add that segment number as a new column in the table.

In [ ]:
final_model = KMeans(n_clusters=3, random_state=0, n_init=10)
segment_labels = final_model.fit_predict(scaled_customers)

# Add the segment number (0, 1 or 2) as a new column.
customers["segment"] = segment_labels

# How many customers fell into each segment?
print("customers per segment:")
print(customers["segment"].value_counts().sort_index())

## Step 6 — profile each segment

Now the business question: *who* is in each segment? We average the original RFM
numbers within each segment. We use the original, unscaled numbers so the averages
come out in real, readable units (days, counts, rupees).

In [ ]:
# Group the table by segment and take the average of each RFM column.
segment_profiles = customers.groupby("segment")[["recency", "frequency", "monetary"]].mean()

print("Average RFM for each segment:")
segment_profiles.round(0)

## Step 7 — read the profiles in plain words

Numbers only matter once someone can act on them, so let us turn the averages into a
human story. We find the segment with the highest average spend and the one with the
lowest, and describe them.

In [ ]:
# Which segment spends the most on average, and which the least.
most_valuable_segment = segment_profiles["monetary"].idxmax()
least_valuable_segment = segment_profiles["monetary"].idxmin()

print("Most valuable segment is segment", most_valuable_segment,
      "- recent, frequent, high spenders (loyal big spenders).")
print("Least valuable segment is segment", least_valuable_segment,
      "- not seen in a long time, rare, low spend (lapsing customers).")
print("The remaining segment sits in the middle (occasional shoppers).")

## Step 8 — visualise the segments with PCA

Our customers live in 3 columns (R, F, M), which is one too many to plot directly. We
use PCA to squash the scaled data down to 2 columns, then colour each dot by its
segment. This is a sanity check: clear, separated colours mean the segments are real
and not an accident.

In [ ]:
# Squash the scaled 3-column data down to 2 columns for plotting.
pca = PCA(n_components=2)
customers_in_2d = pca.fit_transform(scaled_customers)

plt.figure(figsize=(7, 5))
plt.scatter(customers_in_2d[:, 0], customers_in_2d[:, 1],
            c=segment_labels, cmap="viridis", s=30)
plt.xlabel("principal component 1")
plt.ylabel("principal component 2")
plt.title("Customer segments seen in 2D (via PCA)")
plt.show()

variance_kept = pca.explained_variance_ratio_.sum()
print("These 2 components keep", round(variance_kept * 100), "% of the spread.")

### Stretch (optional) — build a plain-language label for every customer

Skip this if you are new to code. Here we attach a readable name to each customer's
row instead of a bare 0, 1 or 2, which is what a real segmentation report would ship.
We rank the segments by average spend, then map each segment number to a name.

In [ ]:
# Rank the segments from lowest to highest average spend.
segments_by_spend = segment_profiles["monetary"].sort_values().index.tolist()

# Give the three ranks readable names.
name_for_segment = {
    segments_by_spend[0]: "lapsing",
    segments_by_spend[1]: "occasional",
    segments_by_spend[2]: "loyal big spender",
}

# Attach the name to every customer row.
customers["segment_name"] = customers["segment"].map(name_for_segment)

print("how many customers carry each name:")
print(customers["segment_name"].value_counts())
print()
print("a few labelled customers:")
customers.head()

## What you just did

You ran a full, real-world segmentation: build RFM features, **scale them**, choose
the number of segments with the elbow and the silhouette, cluster with k-means,
**profile** each segment in plain business terms, and visualise the result with PCA.
The algorithm was the easy part. The value is in naming the segments and deciding
what to do with each one, and that is the part a human owns.